# Lab F — Profile It

**GPU Mastery · CUDA** · run on the **2× RTX 5090** server (WSL2)

Don't guess — measure. This lab walks the profiling ladder hands-on: **nvidia-smi → Nsight Systems → torch.profiler → Nsight Compute**, then runs the **measure → diagnose → fix → measure** loop.

> **Reality check:** `ncu` (Nsight Compute) counters are often **blocked on GeForce cards + WSL**. This lab guards every tool and always gives you a working fallback — `nsys`, `torch.profiler`, and CUDA-event timing carry the day on your 5090.
>
> **On Colab:** `nsys`/`ncu` usually aren't installed — only **Part 1 (dmon)** and **Part 3 (torch.profiler)** will run there.

In [ ]:
import torch, time, shutil, subprocess, os
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
HAS_NSYS = shutil.which("nsys") is not None
HAS_NCU  = shutil.which("ncu")  is not None
print("nsys installed:", HAS_NSYS, " | ncu installed:", HAS_NCU)
!nvidia-smi --query-gpu=name,driver_version --format=csv,noheader

## 1. `nvidia-smi dmon` — watch the GPU while it works

We launch a GPU workload in the background, then capture 8 live samples. Watch the **sm%** (compute busy) column light up during a matmul.

In [ ]:
%%writefile workload.py
import torch, time
a = torch.randn(8192, 8192, device="cuda", dtype=torch.float16)
b = torch.randn_like(a)
end = time.time() + 16
while time.time() < end:
    c = a @ b          # keep the SMs busy
torch.cuda.synchronize()

In [ ]:
proc = subprocess.Popen(["python", "workload.py"])
time.sleep(3)                 # let it spin up
!nvidia-smi dmon -c 8         # 8 one-second samples
proc.terminate(); proc.wait()
print("\nsm% high during the matmul = compute-bound. A memory-bound op would light up mem% instead.")

## 2. Nsight Systems — where does the time go?

`nsys profile --stats=true` runs your program, then prints a **text summary** (no GUI needed): which kernels took the most time, and how much went to memory copies. We profile a workload that mixes **matmul kernels + host→device copies**.

In [ ]:
%%writefile prof_workload.py
import torch
a = torch.randn(4096, 4096, device="cuda", dtype=torch.float16)
b = torch.randn_like(a)
h = torch.randn(4096, 4096, dtype=torch.float16)   # lives on the CPU
for _ in range(20):
    d = h.to("cuda")        # host -> device copy (PCIe)
    c = a @ b               # matmul kernel
torch.cuda.synchronize()

In [ ]:
if HAS_NSYS:
    out = subprocess.run(
        ["nsys", "profile", "--stats=true", "-o", "/tmp/rep", "-f", "true", "python", "prof_workload.py"],
        capture_output=True, text=True)
    txt = out.stdout + out.stderr
    # show the kernel + memory-copy summary tables
    keep = [ln for ln in txt.splitlines() if ln.strip()]
    print("\n".join(keep[-45:]))
else:
    print("nsys not installed — this is the timeline/summary tool. On the server: it should be on PATH with the CUDA toolkit.")

**Read it:** the **CUDA Kernel Summary** shows the matmul (`*gemm*`) eating most of the GPU time; the **Memory Operation Summary** shows the `HtoD` copies. If the copies were a big slice, you'd overlap them with streams — exactly the Act-2 fix.

## 3. `torch.profiler` — the one that always works

For PyTorch code (i.e. LLMs), this is your everyday profiler — per-operation **CUDA time** and **memory**, on any GPU including Colab. We profile a **compute-bound** op (matmul) next to a **memory-bound** one (elementwise add).

In [ ]:
from torch.profiler import profile, ProfilerActivity

a = torch.randn(4096, 4096, device="cuda", dtype=torch.float16)
b = torch.randn_like(a)
for _ in range(5):              # warm up
    _ = a @ b; _ = a + b
torch.cuda.synchronize()

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], profile_memory=True) as prof:
    for _ in range(20):
        c = a @ b               # compute-bound (matmul)
        e = a + b               # memory-bound (elementwise)
    torch.cuda.synchronize()

print(prof.key_averages().table(sort_by="self_cuda_time_total", row_limit=8))

**Read it:** sort by CUDA time. The **matmul** dominates compute time but does a lot of work per byte; the **elementwise add** is cheap in FLOPs but pure memory traffic. This table is how you find the hot op in real LLM code.

## 4. Nsight Compute — and the fallback that always works

`ncu` gives the deepest per-kernel metrics — *if the counters aren't blocked*. We try it; if it's restricted (very likely on a GeForce/WSL box), we fall back to **CUDA-event timing + the roofline**, which needs no special permissions.

In [ ]:
# --- try ncu (may be blocked on GeForce/WSL) ---
if HAS_NCU:
    r = subprocess.run(["ncu", "--set", "basic", "--launch-count", "2", "python", "prof_workload.py"],
                       capture_output=True, text=True)
    if r.returncode == 0 and "Duration" in r.stdout:
        print(r.stdout[-1500:])
    else:
        print("ncu ran but returned limited/no data — likely restricted counters (ERR_NVGPUCTRPERM).")
        print("On GeForce/WSL this is expected. Use the fallback below. ↓")
else:
    print("ncu not installed. Use the fallback below. ↓")

In [ ]:
# --- fallback: measure it yourself (works anywhere, no permissions) ---
def matmul_tflops(n=8192, iters=50):
    a = torch.randn(n, n, device="cuda", dtype=torch.float16); b = torch.randn_like(a)
    for _ in range(10): a @ b
    torch.cuda.synchronize(); t = time.time()
    for _ in range(iters): c = a @ b
    torch.cuda.synchronize()
    return (2 * n**3) / ((time.time() - t) / iters) / 1e12

def bandwidth_gbs(nbytes=2_000_000_000, iters=50):
    x = torch.empty(nbytes // 2, dtype=torch.float16, device="cuda")
    for _ in range(10): y = x.clone()
    torch.cuda.synchronize(); t = time.time()
    for _ in range(iters): y = x.clone()
    torch.cuda.synchronize()
    return 2 * nbytes / ((time.time() - t) / iters) / 1e9

tf = matmul_tflops(); bw = bandwidth_gbs()
ridge = (tf * 1e12) / (bw * 1e9)
print(f"measured peak compute : {tf:8.0f} TFLOP/s")
print(f"measured bandwidth    : {bw:8.0f} GB/s")
print(f"ridge point           : {ridge:8.0f} FLOP/byte")
print("\nThat's a roofline you profiled yourself — classify any kernel's arithmetic intensity against this ridge.")

## 5. The loop, in one experiment: measure → diagnose → fix → measure

Take a **memory-bound** op (matrix × vector, like decode) and a **compute-bound** one (matrix × matrix, like prefill). Measure each as a **% of peak** — the profiler's verdict, by hand.

In [ ]:
def pct_of_peak(op, n=8192, iters=50):
    a = torch.randn(n, n, device="cuda", dtype=torch.float16)
    if op == "matvec":
        x = torch.randn(n, 1, device="cuda", dtype=torch.float16); flops = 2 * n * n; f = lambda: a @ x
    else:
        b = torch.randn(n, n, device="cuda", dtype=torch.float16); flops = 2 * n**3; f = lambda: a @ b
    for _ in range(10): f()
    torch.cuda.synchronize(); t = time.time()
    for _ in range(iters): f()
    torch.cuda.synchronize()
    return flops / ((time.time() - t) / iters) / 1e12

mv = pct_of_peak("matvec"); mm = pct_of_peak("matmul")
print(f"matrix×vector : {mv:7.1f} TFLOP/s  -> {100*mv/tf:4.1f}% of peak   DIAGNOSIS: memory-bound")
print(f"matrix×matrix : {mm:7.1f} TFLOP/s  -> {100*mm/tf:4.1f}% of peak   DIAGNOSIS: compute-bound")
print("\nThe fix for the memory-bound op is NOT more compute — it's batching / better memory reuse.")
print("That is the whole loop: measure the %, diagnose the bound, apply the matching lever, measure again.")

## Reflection (write your answers)

1. In Part 1, what **sm%** did the matmul reach? What would a **memory-bound** kernel show instead?
2. In Part 2, which kernel dominated the nsys summary — and how much time went to **memory copies**?
3. In Part 3, order the ops by CUDA time. Which is compute-heavy, which is pure memory traffic?
4. Did **ncu** work on your 5090, or was it blocked? What did you use instead?
5. In Part 5, what **% of peak** did matrix×vector reach vs matrix×matrix? Which lever would you use to speed up each?

### Cleanup

In [ ]:
!rm -f workload.py prof_workload.py /tmp/rep.nsys-rep /tmp/rep.sqlite
import gc; gc.collect(); torch.cuda.empty_cache()
print("done")